In [27]:
pip install scikit-learn pandas

In [28]:
# Mengimpor semua library yang dibutuhkan sebelum mulai

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [80]:
# Membaca file CSV dan mengambil 2000 baris secara acak agar proses lebih cepat saat latihan

# Read CSV, handle bad lines, and ensure column names are consistent
df = pd.read_csv("IMDB Dataset.csv", sep=',', on_bad_lines='skip', engine='python')

print(f"Columns after initial read: {df.columns.tolist()}")

# Check if 'sentiment;;;;;;' column exists and rename it to 'sentiment'
if 'sentiment;;;;;;' in df.columns:
    df.rename(columns={'sentiment;;;;;;': 'sentiment'}, inplace=True)
    print("Renamed 'sentiment;;;;;;' to 'sentiment'")
elif 'sentiment' not in df.columns:
    # If neither 'sentiment' nor 'sentiment;;;;;;' exist, raise an error or handle accordingly
    # For now, let's assume one of them should exist
    raise KeyError("Neither 'sentiment' nor 'sentiment;;;;;;' column found after reading CSV.")

print(f"Columns after potential renaming: {df.columns.tolist()}")

# Ensure 'sentiment' column is clean and filter out 'none' values BEFORE sampling
df['sentiment'] = df['sentiment'].astype(str).str.strip().str.lower().str.replace(';', '')

print(f"Shape of df after initial read_csv and sentiment cleaning: {df.shape}")
print(f"Value counts for 'sentiment' after initial cleaning:\n{df['sentiment'].value_counts(dropna=False)}")

# Filter out 'none' sentiments from the DataFrame
df_filtered = df[df['sentiment'] != 'none'].copy()

print(f"Shape of df after filtering 'none' sentiments: {df_filtered.shape}")
print(f"Value counts for 'sentiment' after filtering 'none':\n{df_filtered['sentiment'].value_counts(dropna=False)}")

# Sample the DataFrame to 2000 rows, or take all if less than 2000, ensuring shuffling
if len(df_filtered) > 2000:
    processed_df = df_filtered.sample(n=2000, random_state=42)
else:
    # Shuffle the entire DataFrame if it has less than 2000 rows
    processed_df = df_filtered.sample(frac=1.0, random_state=42)

print(f"Shape of processed_df after final sampling: {processed_df.shape}")
print(f"Columns of processed_df after final sampling: {processed_df.columns.tolist()}")
print(f"Value counts for 'sentiment' after final sampling:\n{processed_df['sentiment'].value_counts(dropna=False)}")


Columns after initial read: ['review', 'sentiment;;;;;;']
Renamed 'sentiment;;;;;;' to 'sentiment'
Columns after potential renaming: ['review', 'sentiment']
Shape of df after initial read_csv and sentiment cleaning: (23073, 2)
Value counts for 'sentiment' after initial cleaning:
sentiment
none        22995
positive       48
negative       30
Name: count, dtype: int64
Shape of df after filtering 'none' sentiments: (78, 2)
Value counts for 'sentiment' after filtering 'none':
sentiment
positive    48
negative    30
Name: count, dtype: int64
Shape of processed_df after final sampling: (78, 2)
Columns of processed_df after final sampling: ['review', 'sentiment']
Value counts for 'sentiment' after final sampling:
sentiment
positive    48
negative    30
Name: count, dtype: int64


In [81]:
#  preprocessing agar kata "Good" dan "good" dianggap sama oleh model

processed_df['review'] = processed_df['review'].str.lower()

In [82]:
# Label Encoding
# The processed_df should now contain only 'positive' or 'negative' sentiments after preprocessing in the previous cell

print(f"Shape of processed_df before label mapping: {processed_df.shape}")
print(f"Value counts for 'sentiment' before label mapping:\n{processed_df['sentiment'].value_counts(dropna=False)}")

processed_df['label'] = processed_df['sentiment'].map({'positive': 1, 'negative': 0})

print(f"Value counts for 'label' after mapping (including NaNs):\n{processed_df['label'].value_counts(dropna=False)}")
print(f"Number of NaNs in 'label' after mapping: {processed_df['label'].isnull().sum()}")
print(f"Shape of processed_df after label creation: {processed_df.shape}")

Shape of processed_df before label mapping: (78, 2)
Value counts for 'sentiment' before label mapping:
sentiment
positive    48
negative    30
Name: count, dtype: int64
Value counts for 'label' after mapping (including NaNs):
label
1    48
0    30
Name: count, dtype: int64
Number of NaNs in 'label' after mapping: 0
Shape of processed_df after label creation: (78, 3)


In [83]:
#  TF IDF Mengubah teks ulasan menjadi angka (vektor)
processed_df.dropna(subset=['review', 'label'], inplace=True)
print(f"Shape of processed_df after dropping NaNs: {processed_df.shape}")
print(f"Value counts for processed_df['label'] after dropping NaNs:\n{processed_df['label'].value_counts(dropna=False)}")

tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(processed_df['review']).toarray()
y = processed_df['label'].values.astype(int)

print(f"Shape of X after TFIDF: {X.shape}")
print(f"Shape of y after label conversion: {y.shape}")

Shape of processed_df after dropping NaNs: (78, 3)
Value counts for processed_df['label'] after dropping NaNs:
label
1    48
0    30
Name: count, dtype: int64
Shape of X after TFIDF: (78, 543)
Shape of y after label conversion: (78,)


In [84]:
# SPLIT DATA Membagi data menjadi data latih (80%) dan data uji (20%)
# random_state=42 agar pembagian selalu sama setiap dijalankan

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (62, 543)
Shape of y_train: (62,)
Shape of X_test: (16, 543)
Shape of y_test: (16,)


In [85]:
# latih data logistic regression
model = LogisticRegression()
model.fit(X_train, y_train)

LogisticRegression()

In [86]:
# evaluasi model

y_pred = model.predict(X_test)
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[0 7]
 [0 9]]

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         7
           1       0.56      1.00      0.72         9

    accuracy                           0.56        16
   macro avg       0.28      0.50      0.36        16
weighted avg       0.32      0.56      0.40        16



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
